In [ ]:
# !pip install --upgrade setuptools packaging
# !pip install -e ..

In [ ]:
# !conda install -c conda-forge ta-lib -y

In [ ]:
import databento as db
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime, timedelta
from tqdm import tqdm
import pytz

In [ ]:

from blockhouse_ml.utils.macro_model import MetaLearner
from blockhouse_ml.options.utils.data_handler import DataProcessor, InferenceDataHandler
from blockhouse_ml.options.utils.macro_model_utils import MacroTraderModel
from blockhouse_ml.options.utils.fetch_merge_data import PolygonClient
from blockhouse_ml.options.utils.env import TradingEnvironment

In [ ]:
databento_api_key = "db-eKU7cAt4iTryxUbycEY7REuXXkwcU"

In [ ]:
# Initialize the model directory, where the models will be saved
UNET_MODEL_DIR = '../OptionsUnetModels'
os.makedirs(UNET_MODEL_DIR, exist_ok=True)

TAB_MODEL_DIR = '../OptionsTabModels'
os.makedirs(TAB_MODEL_DIR, exist_ok=True)

# Initialize the data directory, where the data will be stored
data_dir = '../OptionsData'
os.makedirs(data_dir, exist_ok=True)

# Initialize the MetaLearner
meta = MetaLearner()

# Initialize the MacroTraderModel
macro_trader_unet = MacroTraderModel(UNET_MODEL_DIR)

macro_trader_tab = MacroTraderModel(TAB_MODEL_DIR)

data_client = PolygonClient(save_dir=data_dir)

# Initialize the data processor
data_processor = DataProcessor()

# Inference data handler
inference_data_handler = InferenceDataHandler()

# Define the forecast steps
forecast_steps = {
    'open': (360, '1T'),
    'high': (360, '1T'),
    'low': (360, '1T'),
    'close': (360, '1T'),
    'volatility': (360, '1T'),
    'volume': (360, '1T'),
    'transaction_cost': (360, '1T')
}



# Define the start and end dates
start_time = '2024-07-01'
end_time = '2024-08-16'

# List of Companies based on Market Capitalization
large_cap_companies = []#['AAPL', 'CSCO', 'MCD', 'IBM', 'AMZN', 'TSLA', 'PFE', 'MS','MSFT','NVDA']
mid_cap_companies = [] #['AEG', 'NICE', 'NLY', 'ONTO', 'PSN', 'SAIA', 'OWL','PNW','TWLO','HAS']
small_cap_companies = []#['NVAX','AMC','WOLF','IREN','SEDG', 'UPWK','FSLY','BMBL','ARRY']

In [ ]:
def get_expected_price(contract, timestamp):
    # Initialize the client with your API key
    # client = RESTClient(api_key)

    # Convert timestamp to nanoseconds
    end_timestamp = int(timestamp.timestamp() * 1_000_000_000)
    start_timestamp = end_timestamp - (60 * 1_000_000_000)  # Subtract 1 minute from the timestamp for start time

    client = db.Historical(databento_api_key)

    data = client.timeseries.get_range(
        dataset="OPRA.PILLAR", # for stocks which require MBO
        schema="mbp-1",
        stype_in="raw_symbol",
        symbols=[contract],
        start=start_timestamp,
        end=end_timestamp,

    )
    df = data.to_df()
    # print(df.head())

    # # Make the API call
    # quotes = client.list_quotes(
    #     contract,
    #     timestamp_gte=start_timestamp,
    #     timestamp_lte=end_timestamp,
    #     limit=50000  # Adjust based on your needs
    # )

    # # Initialize variables to store maximum bid price and sizes
    max_bid_price = float('-inf')
    bid_sizes = []
    ask_sizes = []
    bid_prices = []
    ask_prices = []
    quotes = []
    for index, row in df.iterrows():
        bid_prices.append(row['bid_px_00'])
        bid_sizes.append(row['bid_sz_00']*100)
        ask_sizes.append(row['ask_sz_00']*100)
        ask_prices.append(row['ask_px_00'])
        if row['bid_px_00'] > max_bid_price:
            max_bid_price = row['bid_px_00']
        # quotes.append(row['bid_price'], row['bid_size'], row['ask_price'], row['ask_size']))
    # # Process the results to find the maximum bid price and collect sizes
    # for quote in quotes:
    #     # print(pd.to_datetime(start_timestamp, unit='ns'), pd.to_datetime(end_timestamp, unit='ns'), quote)
    #     if quote.bid_price > max_bid_price:
    #         max_bid_price = quote.bid_price
    #     bid_prices.append(quote.bid_price)
    #     bid_sizes.append(quote.bid_size*100)
    #     ask_sizes.append(quote.ask_size*100)
    #     ask_prices.append(quote.ask_price)

    # Check if max_bid_price was updated, otherwise handle no data case
    if max_bid_price == float('-inf'):
        print(f"No bid prices found for the given timestamp: {timestamp}")
        print(f"{start_timestamp}, {end_timestamp}, {contract}")
        return None, None, None, None, None
    bid_prices = np.array(bid_prices)#[~np.isnan(bid_prices)]
    bid_sizes = np.array(bid_sizes)#[~np.isnan(bid_sizes)]
    ask_sizes = np.array(ask_sizes)#[~np.isnan(ask_sizes)]
    ask_prices = np.array(ask_prices)#[~np.isnan(ask_prices)]

    concat_arr = np.vstack((bid_prices, bid_sizes, ask_sizes, ask_prices))
    concat_arr_without_nan = concat_arr[:,~np.isnan(concat_arr).any(axis=0)]

    bid_prices = concat_arr_without_nan[0]
    bid_sizes = concat_arr_without_nan[1]
    ask_sizes = concat_arr_without_nan[2]
    ask_prices = concat_arr_without_nan[3] 

    return max_bid_price, bid_sizes, ask_sizes, bid_prices, ask_prices


def calculate_vwap(bid_prices, bid_sizes):
    # VWAP calculation: sum(price * size) / sum(size)
    vwap = np.sum(bid_prices * bid_sizes) / np.sum(bid_sizes)
    return vwap

In [ ]:
import pytz
quicker_training_filepath = f'{data_dir}/options-train-data.csv'
output_path = f'{data_dir}/macro-trader-training-data.csv'
ticker= 'AAPL'
## User specific Option data
user_data = {
    "option_type" : "C",
    "strike_price" : 100,
    "maturity_date" : "240823"
}

est_tz = pytz.timezone('America/New_York')

processed_data = pd.read_csv(quicker_training_filepath)
processed_data['contract'] = f"{ticker}  {user_data['maturity_date']}C00100000"
processed_data['datetime'] = pd.to_datetime(processed_data['timestamp'])
# processed_data['timestamp'] = processed_data['datetime']
processed_data['VWAP'] = processed_data['VWAP_bid']
processed_data = processed_data[processed_data['datetime'].dt.dayofweek < 5]
processed_data.set_index('datetime', inplace=True)
# print(processed_data)
processed_data = processed_data.between_time('13:30', '20:00')

processed_data.reset_index(inplace=True)
processed_data =data_processor.process_data(processed_data,option_type=user_data['option_type'],strike_price=user_data['strike_price'],n_jobs=2)
# processed_data['datetime'] = processed_data['datetime']
print(processed_data.columns)
# processed_data.to_csv(output_path)
processed_data

In [ ]:
def get_training_dataset(idx_set, data, contract):
    res = {"datetime" : [], "spread_cost" : [], "actual_price" : [], "expected_price":[]}
    for current_step in idx_set:
        print(f"Current step: {current_step}",file=sys.stderr)
        current_timestamp = data['datetime'].iloc[current_step]
        timestamp = pd.to_datetime(current_timestamp)#.replace(tzinfo=est_tz)

        # Fetch expected price, bid_sizes, ask_sizes, and bid_prices from the Polygon API
        expected_price, bid_sizes, ask_sizes, bid_prices, ask_prices = get_expected_price(contract, timestamp)
        if expected_price is None:
            # print("Expected price couldn't be fetched, returning no reward.")
            continue
        # print("expected_price:",expected_price)
        spread_cost = np.min(ask_prices) - np.max(bid_prices)
        actual_price = calculate_vwap(bid_prices, bid_sizes)
        
        res["datetime"].append(data.iloc[current_step]["datetime"])
        res["spread_cost"].append(spread_cost)
        res["actual_price"].append(actual_price)
        res["expected_price"].append(expected_price)
        # data.loc[data.index[current_step],"spread_cost"] = spread_cost
        # data.loc[data.index[current_step],"actual_price"] = actual_price
        # print(f"{spread_cost}, {actual_price}",file=sys.stderr)
        # print(f"{data.iloc[current_step]['spread_cost']}, {data.index[current_step]}",file=sys.stderr)

    d = pd.DataFrame(res)
    # d.reset_index()
    return d



In [ ]:
from joblib import Parallel, delayed
from more_itertools import chunked
import sys
n_jobs = -1
temp_data = processed_data.copy()
# if 'spread_cost' not in temp_data.columns:
#     temp_data['spread_cost'] = np.nan
# if 'actual_price' not in temp_data.columns:
#     temp_data['actual_price'] = np.nan
chunk_set = chunked(range(len(temp_data)), int(len(temp_data)/32))
# %timeit 
results = Parallel(n_jobs=n_jobs)(
    [delayed(get_training_dataset)(idx_set, temp_data, temp_data['contract'].iloc[0]) for idx_set in tqdm(chunk_set, desc="Processing rows")])
data = pd.concat(results)
f_data= pd.merge(temp_data, data, on="datetime",how="inner")

In [ ]:
output_path = f'{data_dir}/macro-trader-training-data-parallel.csv'
f_data.to_csv(output_path)

In [ ]:
!pip install "modin[all]"
!pip install "bokeh!=3.0.*,>=2.4.2"

In [ ]:
def get_training_dataset(row):
    # for current_step in idx_set:
        # print(f"Current step: {current_step}")
    current_timestamp = row['datetime']
    contract = row['contract']
    timestamp = pd.to_datetime(current_timestamp)#.replace(tzinfo=est_tz)


    # Fetch expected price, bid_sizes, ask_sizes, and bid_prices from the Polygon API
    expected_price, bid_sizes, ask_sizes, bid_prices, ask_prices = get_expected_price(contract, timestamp)
    if expected_price is None:
        # print("Expected price couldn't be fetched, returning no reward.")
        row["spread_cost"] = np.nan
        row["actual_price"] = np.nan
        return row
    spread_cost = np.min(ask_prices) - np.max(bid_prices)
    actual_price = calculate_vwap(bid_prices, bid_sizes)

    row["spread_cost"] = spread_cost
    row["actual_price"] = actual_price

    return row


In [ ]:
DASK_RUNNING = False
import os
os.environ["MODIN_ENGINE"] = "dask"  # Modin will use Dask
if not DASK_RUNNING:
    from dask.distributed import Client, LocalCluster
    cluster = LocalCluster()  # Launches a scheduler and workers locally
    client = Client(cluster)  # Connect to distributed cluster and override default
    print(f"Started cluster at {cluster.dashboard_link}")
    DASK_RUNNING = True
import modin.pandas as pd
from modin.config import ProgressBar
ProgressBar.enable()
data = pd.DataFrame(processed_data)
if 'spread_cost' not in data.columns:
    data['spread_cost'] = np.nan
if 'actual_price' not in data.columns:
    data['actual_price'] = np.nan
# Use tqdm to show progress
# tqdm.pandas(desc="Processing rows")
data = data.apply(get_training_dataset, axis=1)
data.to_csv(output_path)
print("Modin results:\n", sample.tail(5))